# Hooke's Law — Mass on a Spring

A block of mass $m$ sits on a frictionless floor, attached to a spring of stiffness $k$. It is pulled a distance $x_0$ from equilibrium and released.

The spring obeys **Hooke's law**:

$$F = -kx$$

— the further the spring is stretched, the harder it pulls back, and the force always points towards equilibrium. Since $F = ma$, the acceleration is $a = -\dfrac{k}{m}x$, which produces **simple harmonic motion**:

$$x(t) = x_0\cos(\omega t), \qquad \omega = \sqrt{\frac{k}{m}}, \qquad T = 2\pi\sqrt{\frac{m}{k}}$$

- **Left:** the block oscillating (the dotted line marks equilibrium; the ghost block marks the release point). The black arrow is the spring force $F = -kx$ — watch it shrink to **zero at equilibrium** and grow in proportion to the stretch.
- **Right:** the energies plotted **against position**. The dots slide back and forth along their curves in time with the block.

The three curves:

- **Blue — potential energy**, $PE = \tfrac{1}{2}kx^2$: greatest at the extremes, zero at equilibrium.
- **Green — kinetic energy**, $KE = \tfrac{1}{2}mv^2$: zero at the extremes, greatest at equilibrium.
- **Red — total energy**, $E = \tfrac{1}{2}kx_0^2$: a flat line — energy is **conserved** (no friction).

At the release point it's all PE; at equilibrium it's all KE; in between, the energy is continuously swapping between the two.

**Things to try:**

- Heavier mass → slower oscillation: $T = 2\pi\sqrt{m/k}$.
- Stiffer spring (bigger $k$) → faster oscillation, and more energy for the same stretch.
- Bigger stretch $x_0$ → more energy, but the **period is unchanged** — $T$ doesn't depend on the amplitude.

**Hooke's-law check:** watch the force readout as the block moves — the force doubles when the stretch doubles, and flips direction as the block crosses equilibrium. A graph of $F$ against $x$ would be a straight line through the origin with slope $-k$.


In [1]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle, FancyArrowPatch

from matplotlib.animation import FuncAnimation

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    %pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display, HTML


# ------------------------------------------------------------
# The physics
# ------------------------------------------------------------

def omega(m, k):
    """Angular frequency of a mass on a spring: w = sqrt(k/m)."""
    return np.sqrt(k / m)


def period(m, k):
    """T = 2*pi*sqrt(m/k) — independent of the amplitude."""
    return 2 * np.pi * np.sqrt(m / k)


def energies(k, x, x0):
    """PE, KE and total E at displacement x, released from rest at x0."""
    E = 0.5 * k * x0**2
    PE = 0.5 * k * x**2
    KE = E - PE
    return PE, KE, E


# ------------------------------------------------------------
# Drawing
# ------------------------------------------------------------

def spring_line(x_start, x_end, y, n_coils=12, amp=0.09):
    """(xs, ys) for a zig-zag spring from (x_start, y) to (x_end, y)."""
    xs = np.linspace(x_start, x_end, 2 * n_coils + 1)
    ys = np.full_like(xs, y)
    ys[1:-1:2] = y + amp
    ys[2:-1:2] = y - amp
    return xs, ys


def build_spring_figure(m, k, x0):
    """Two panels: the block on a spring (left) and the energy plot (right)."""
    E = 0.5 * k * x0**2

    bw, bh = 0.55, 0.55                       # block width / height
    x_wall = -(x0 + 1.15)                     # wall position
    y_s = 0.5 * bh                            # spring height (block mid-height)
    y_F = bh + 0.62                           # height of the force arrow

    fig, (axs, axe) = plt.subplots(
        1, 2, figsize=(11, 4.6), gridspec_kw=dict(width_ratios=[1, 1.35]))

    # ---------------- left: block on a spring ----------------
    axs.set_xlim(x_wall - 0.25, x0 + 1.05)
    axs.set_ylim(-0.45, 1.75)

    # wall (hatched) and frictionless floor
    axs.plot([x_wall, x_wall], [0, 1.15], color="black", linewidth=3)
    for yh in np.linspace(0.12, 1.08, 8):
        axs.plot([x_wall, x_wall - 0.14], [yh, yh + 0.14],
                 color="black", linewidth=1)
    axs.plot([x_wall, x0 + 1.05], [0, 0], color="black", linewidth=2)

    # equilibrium marker and ghost block at the release point
    axs.plot([0, 0], [0, 1.15], ls=":", color="gray", linewidth=1.2, zorder=1)
    axs.text(0, -0.16, "equilibrium (x = 0)", ha="center", va="top",
             fontsize=9, color="dimgray")
    ghost = Rectangle((x0 - bw / 2, 0), bw, bh, facecolor="none",
                      edgecolor="gray", ls=":", zorder=1)
    axs.add_patch(ghost)
    axs.text(x0, bh + 0.07, "release", ha="center", va="bottom",
             fontsize=9, color="dimgray")

    # spring, block, mass label
    spring = axs.plot([], [], color="black", linewidth=1.5, zorder=2)[0]
    block = Rectangle((x0 - bw / 2, 0), bw, bh, facecolor="lightblue",
                      edgecolor="black", linewidth=2, zorder=3)
    axs.add_patch(block)
    mass_text = axs.text(x0, bh / 2, f"{m:g} kg", ha="center", va="center",
                         fontsize=12, zorder=4)

    # spring force arrow (F = -kx) floating above the block
    force_arrow = FancyArrowPatch((x0, y_F), (x0, y_F), arrowstyle="-|>",
                                  mutation_scale=20, linewidth=2.5,
                                  color="black", zorder=4)
    axs.add_patch(force_arrow)
    force_text = axs.text(x0, y_F + 0.06, "", ha="center", va="bottom",
                          fontsize=11, fontweight="bold")

    info_s = axs.text(0.03, 0.96, "", transform=axs.transAxes,
                      ha="left", va="top", fontsize=11,
                      bbox=dict(facecolor="white", alpha=0.85,
                                edgecolor="lightgray"))

    axs.set_title(f"m = {m:g} kg,  k = {k:g} N/m,  released from x₀ = {x0:g} m",
                  fontsize=12)
    axs.set_xticks([])
    axs.set_yticks([])
    for side in ("left", "right", "top", "bottom"):
        axs.spines[side].set_visible(False)

    # ---------------- right: energy vs position ----------------
    xs = np.linspace(-x0, x0, 200)
    PE_curve = 0.5 * k * xs**2
    KE_curve = E - PE_curve

    axe.plot(xs, PE_curve, color="tab:blue", linewidth=2.2)
    axe.plot(xs, KE_curve, color="tab:green", linewidth=2.2)
    axe.plot([-x0, x0], [E, E], color="tab:red", linewidth=2.2)

    axe.text(0, 0.06 * E, "PE = ½kx²", color="tab:blue", fontsize=12,
             fontweight="bold", ha="center", va="center")
    axe.text(0, 0.88 * E, "KE", color="tab:green", fontsize=12,
             fontweight="bold", ha="center", va="center")
    axe.text(x0 - 0.03 * x0, 1.05 * E, "E = PE + KE", color="tab:red",
             fontsize=12, fontweight="bold", ha="right", va="center")

    pe_dot = axe.plot([], [], "o", markersize=11, color="tab:blue",
                      markeredgecolor="black", zorder=5)[0]
    ke_dot = axe.plot([], [], "o", markersize=11, color="tab:green",
                      markeredgecolor="black", zorder=5)[0]
    e_dot = axe.plot([], [], "o", markersize=11, color="tab:red",
                     markeredgecolor="black", zorder=5)[0]
    vline = axe.axvline(x0, color="gray", linewidth=1, alpha=0.6, zorder=2)

    info_e = axe.text(0.5, 0.97, "", transform=axe.transAxes,
                      ha="center", va="top", fontsize=11,
                      bbox=dict(facecolor="white", alpha=0.85,
                                edgecolor="lightgray"))

    pad = max(0.12 * x0, 0.05)
    axe.set_xlim(-x0 - pad, x0 + pad)
    axe.set_ylim(0, 1.18 * E)
    axe.set_xlabel("Displacement from equilibrium x (m)")
    axe.set_ylabel("Energy (J)")
    axe.set_title(f"Energy vs position — total is always {E:.3g} J",
                  fontsize=12)
    for side in ("right", "top"):
        axe.spines[side].set_visible(False)

    fig.tight_layout()

    artists = dict(
        spring=spring, block=block, mass_text=mass_text,
        force_arrow=force_arrow, force_text=force_text, info_s=info_s,
        pe_dot=pe_dot, ke_dot=ke_dot, e_dot=e_dot,
        vline=vline, info_e=info_e,
    )
    geom = dict(bw=bw, bh=bh, x_wall=x_wall, y_s=y_s, y_F=y_F)
    return fig, artists, geom


def set_state(art, geom, m, k, x0, x, v):
    """Move every artist to match displacement x and velocity v."""
    # block, mass label, spring
    art["block"].set_x(x - geom["bw"] / 2)
    art["mass_text"].set_x(x)
    xs_sp, ys_sp = spring_line(geom["x_wall"], x - geom["bw"] / 2,
                               geom["y_s"])
    art["spring"].set_data(xs_sp, ys_sp)

    # spring force F = -kx: arrow above the block, length proportional to |x|
    F = -k * x
    a = F / m
    arrow_len = 0.9 * abs(x)
    if arrow_len > 1e-9:
        direction = -np.sign(x)               # force points back to equilibrium
        art["force_arrow"].set_visible(True)
        art["force_arrow"].set_positions(
            (x, geom["y_F"]), (x + direction * arrow_len, geom["y_F"]))
        art["force_text"].set_position(
            (x + direction * arrow_len / 2, geom["y_F"] + 0.06))
        art["force_text"].set_text(f"F = −kx = {F:+.1f} N")
    else:
        art["force_arrow"].set_visible(False)
        art["force_text"].set_text("")

    # energy dots
    PE, KE, E = energies(k, x, x0)
    art["pe_dot"].set_data([x], [PE])
    art["ke_dot"].set_data([x], [KE])
    art["e_dot"].set_data([x], [E])
    art["vline"].set_xdata([x])

    art["info_s"].set_text(
        f"x = {x:+.2f} m\nv = {v:+.2f} m/s\nF = −kx = {F:+.1f} N\n"
        f"a = F/m = {a:+.2f} m/s²")
    art["info_e"].set_text(
        f"PE = {PE:.3g} J      KE = {KE:.3g} J      PE + KE = {PE + KE:.3g} J")


def show_release(m, k, x0):
    """Static preview: block at the release point, dots at the start."""
    fig, art, geom = build_spring_figure(m, k, x0)
    set_state(art, geom, m, k, x0, x0, 0.0)
    display(fig)
    plt.close(fig)


def spring_animation(m, k, x0, n_frames=64):
    """In-browser animation: one full oscillation, looping in real time."""
    fig, art, geom = build_spring_figure(m, k, x0)
    T = period(m, k)
    w = omega(m, k)
    ts = np.linspace(0, T, n_frames, endpoint=False)
    xs = x0 * np.cos(w * ts)
    vs = -x0 * w * np.sin(w * ts)
    interval = min(max(1000.0 * T / n_frames, 30), 80)   # ~ real time

    def update(i):
        set_state(art, geom, m, k, x0, xs[i], vs[i])
        return tuple(art.values())

    anim = FuncAnimation(fig, update, frames=n_frames,
                         interval=interval, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def summary(m, k, x0):
    E = 0.5 * k * x0**2
    v_max = x0 * omega(m, k)
    T = period(m, k)
    return (f"<b>Hooke's law:</b> F = −kx &nbsp;→&nbsp; strongest pull at "
            f"release, F_max = kx₀ = ({k:g})({x0:g}) = <b>{k * x0:g} N</b><br>"
            f"<b>E = ½kx₀²</b> = ½({k:g})({x0:g})² = <b>{E:.3g} J</b> "
            f"&nbsp;·&nbsp; at equilibrium it's all KE, "
            f"v_max = x₀√(k/m) = <b>{v_max:.3g} m/s</b><br>"
            f"Period T = 2π√(m/k) = <b>{T:.2f} s</b> — the same no matter "
            f"how far you pull it")


In [2]:
# ============================================================
# Mass on a spring: set mass, spring constant, stretch, then Run
# ============================================================

mass_slider = widgets.FloatSlider(
    value=2.0, min=0.5, max=5.0, step=0.5,
    description="m (kg):", continuous_update=False
)
k_slider = widgets.FloatSlider(
    value=20.0, min=5.0, max=50.0, step=5.0,
    description="k (N/m):", continuous_update=False
)
stretch_slider = widgets.FloatSlider(
    value=0.6, min=0.2, max=1.0, step=0.1,
    description="x₀ (m):", continuous_update=False
)

run_button = widgets.Button(description="Run", button_style="primary", icon="play")
reset_button = widgets.Button(description="Reset")
value_label = widgets.HTML()
sim_output = widgets.Output(layout=widgets.Layout(width="1000px"))


def show_preview():
    value_label.value = summary(mass_slider.value, k_slider.value,
                                stretch_slider.value)
    show_release(mass_slider.value, k_slider.value, stretch_slider.value)


def run_clicked(button):
    value_label.value = summary(mass_slider.value, k_slider.value,
                                stretch_slider.value)
    with sim_output:
        sim_output.clear_output(wait=True)
        display(spring_animation(mass_slider.value, k_slider.value,
                                 stretch_slider.value))


def reset_clicked(button):
    with sim_output:
        sim_output.clear_output(wait=True)
        show_preview()


def slider_changed(change):
    with sim_output:
        sim_output.clear_output(wait=True)
        show_preview()


run_button.on_click(run_clicked)
reset_button.on_click(reset_clicked)
mass_slider.observe(slider_changed, names="value")
k_slider.observe(slider_changed, names="value")
stretch_slider.observe(slider_changed, names="value")

controls = widgets.VBox([
    widgets.HTML("<h3>Controls</h3>"),
    mass_slider,
    k_slider,
    stretch_slider,
    widgets.HBox([run_button, reset_button]),
    value_label
], layout=widgets.Layout(width="330px"))

display(widgets.HBox([controls, sim_output],
                     layout=widgets.Layout(align_items="flex-start", gap="18px")))

with sim_output:
    show_preview()
